### 전이 학습
이미 학습된 모델의 지식을 새로운 문제에 재사용하는 방법

<ul>
    <li>처음부터 학습시키기 위해서는 수만 ~ 수백만 장의 데이터와 긴 학습 시간이 필요함</li>
    <li>처음부터 학습으로 인하여 데이터 부족, 과적합, 학습 시간 증가 문제가 발생할 수 있음</li>
    <li>이미 대규모 데이터셋으로 학습된 모델을 가져와 사용해 적은 데이터로도 높은 성능을 낼 수 있음</li>
</ul>

#### 1. 전이학습 원리

<ul>
    <li>기존 학습 : Image Net (1000개 클래스) → ResNet50 → 특징 추출 능력 획득</li>
    <li>새로운 문제 : 고양이 vs 강아지 → 기존 ResNet50 활용 → 마지막 출력층만 교체 → 재학습</li>
</ul>

#### 2. 대표적인 사전학습 모델

VGG16
<ul>
    <li>구조가 단순 (Conv Conv Pooling Conv Conv Pooling ...)</li>
    <li>특징 : 이해하기 쉬움, 파라미터 수 많음, 속도 느림</li>
</ul>

ResNet
<ul>
    <li>가장 많이 사용하는 모델</li>
    <li>입력 → Conv → Conv → + (입력값) → 출력</li>
    <li>Skip Connection 또는 Residual Connection을 사용함</li>
    <li>매우 깊은 네트워크 가능, Gradient Vanishing 문제 감소</li>
    <li>대표적인 모델 : ResNet18, ResNet34, ResNet50, ResNet101</li>
</ul>

MobileNet
<ul>
    <li>모바일용 모델</li>
    <li>특징 : 크기 작음, 속도 빠름, 스마트폰 배포 적합</li>
    <li>사용 예 : 안드로이드 앱, 임베디드 시스템</li>
</ul>

Efficientnet
<ul>
    <li>최근에 많이 사용되는 모델</li>
    <li>특징 : 적은 파라미터, 높은 정확성, 효율성 우수</li>
    <li>버전 : B0, B1, B2, ... , B7 (숫자가 커질수록 성능 증가)</li>
</ul>

#### 3. Feature Extraction
<ul>
    <li>가장 쉬운 전이학습 방법</li>
    <li>기존 모델 가중치를 고정</li>
    <li>ResNet50 → 특징 추출 → 새 분류기(Dense) 에서 Dense Layer만 학습</li>
    <li>장점 : 학습 빠름, 데이터가 적어도 가능, 과적합 감소</li>
    <li>단점 : 새로운 데이터와 차이가 크면 성능 제한</li>
</ul>

In [ ]:
# Feature Extraction 예시
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

moel.keras.Sequential([baseModel, tf.keras.layers.GlobalAveragePooling2D(), tf.keras.layers.Dense(10, activatio='softmax')])

#### 4. Find Tuning
Feature Extraction보다 한 단계 발전된 방법

<ul>
    <li>개념 : 사전학습 모델의 일부 층도 같이 학습하는 방법</li>
    <li>ResNet50 → 상위 몇 개 Layer만 학습 → 새 Dense Layer 학습</li>
</ul>

In [ ]:
# Find Tuning 예제
base_model.trainable = True

# 마지막 20개 층만 학습
for layer in base_model.layers[:-20]:
    layer.trainable = False

<ul>
    <li>장점 : 눈, 귀, 코와 같은 특징은 이미 알고 있기 때문에 자신만의 특징으로 조금 수정할때 좋음</li>
</ul>

In [ ]:
# 보통 Learning Rate가 크면 이미 기존 자식을 파괴하는 Catastrophic Forgetting 현상이 발생하기 때문에 Learning Rate를 작게 구성
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5)

#### 5. CIFAR-10에서 전이 학습

<ul>
    <li>Conv2D Conv2D Conv2D를 통해서 직접 학습하는 방법은 정확도가 75 ~ 85%로 낮고, 학습하는데 오래 걸림</li>
    <li>ResNet50을 이용하면 정확도가 90% 이상이며, 학습이 매우 빠름</li>
    <li>실무에서 가장 많이 사용하는 패턴</li>
    <ul>
        <li>1. EfficientNet50 불러오기</li>
        <li>2. base_model.trainable = False</li>
        <li>3. Dense Layer 추가</li>
        <li>4. Feature Extraction 학습</li>
        <li>5. 마지막 20 ~ 30개 Layer Fine Tuning</li>
        <li>6. Learning rate 1e-5로 재학습</li>
    </ul>
</ul>